# Sionna RT — USD / Omniverse Scene Builder (London)
## Full-pipeline equivalent of `sionna019_scene_builder_london.ipynb` — OSM + EA LiDAR + USD export

When `USE_OSM_BUILDINGS=True` (default), this notebook downloads real London building footprints
and heights from OpenStreetMap, drapes them onto EA LiDAR DTM terrain, adds OSM roads/vegetation/water,
assigns ITU-R P.2040-2 materials, and writes the same output files as the OSM scene builder.

When `DEMO_MODE=False` and a real `.usd` file is available, it reads mesh geometry + materials
directly from the USD stage instead of OSM (e.g. a photogrammetric export from Omniverse or Blender).

---

## Build Sequence

| Step | Cell | Description | Skip when |
|------|------|-------------|----------|
| 1 | **CELL 0** | Configuration | Never |
| 2 | **CELL 1** | Imports | Once per session |
| 3 | **CELL 2** | OSM buildings (or USD load) | Never after scene change |
| 4 | **CELL 2b** | EA LiDAR DTM terrain + drape | `dem.tif` exists + no new buildings |
| 5 | **CELL 2c** | OSM roads, vegetation, water | Never after scene change |
| 6 | **CELL 2d** | Export `.usda` (Omniverse round-trip) | `EXPORT_USD=False` |
| 7 | **CELL 3** | Map materials → ITU-R names | Never |
| 8 | **CELL 3b** | NVIDIA NIM visual classification | `USE_NVIDIA_MATERIAL_CLASSIFICATION=False` |
| 9 | **CELL 4** | Write per-mesh PLYs | Never |
| 10 | **CELL 5** | Write `scene.xml` (0.19) + `scene_with_full.xml` (2.0) | Never |
| 11 | **CELL 6** | 2D top-down preview | Optional |
| 12 | **CELL 7** | Load into Sionna RT, sanity-check | Optional |

**Outputs** (in `~/sionna_rt/london_omniverse_usd/scene_usd/`):
- `scene_with_full.xml` — Sionna 2.0 format → used by `sionna2_915mhz_dem_simulation_london.ipynb`
- `scene.xml` — Sionna 0.19 format → used by `sionna019_differentiable_rt_fixed.ipynb`
- `scene_built.usda` — USDA ASCII for Omniverse / USD View
- `meshes/terrain.ply` — EA LiDAR terrain mesh
- `meshes/usd_*.ply` — per-mesh building/road/vegetation PLYs


## CELL 0 — Configuration

Set `USE_OVERTURE_BUILDINGS=True` (default) to use **Overture Maps** building footprints
— free, no API key, better accuracy than OSM, backed by Amazon/Meta/Microsoft/TomTom.

Set `USE_OSM_BUILDINGS=True` to fall back to OpenStreetMap.
Set `DEMO_MODE=False` + `USD_FILE=...` to load from a real USD/Omniverse export.

In [ ]:
# ============================================================
# CELL 0 — CONFIG  (edit this block only)
# ============================================================
import os

# Every USE_*/INCLUDE_* toggle below can be overridden from the shell without
# editing this file, e.g. `USE_NVIDIA_MATERIAL_CLASSIFICATION=True jupyter ...` --
# the hardcoded value alongside each one is just the default when no env var is set.
def _env_bool(name, default):
    v = os.environ.get(name)
    return default if v is None else v.strip().lower() in ('1', 'true', 'yes', 'on')

SCENARIO_NAME = 'london_omniverse_usd'
CITY_NAME     = 'London'

# ── Demo mode ─────────────────────────────────────────────────────────────────
# True  -> build a small synthetic in-memory scene (no usd-core, no real file
#          needed) so the rest of the notebook is runnable today.
# False -> load USD_FILE for real via pxr/usd-core.
DEMO_MODE = _env_bool('DEMO_MODE', True)
USD_FILE  = '/path/to/your/omniverse_scene.usd'   # only used when DEMO_MODE=False

BASE_DIR  = os.environ.get('RT_BASE_DIR', os.path.join(os.path.expanduser('~'), 'sionna_rt', SCENARIO_NAME))
SCENE_DIR = os.path.join(BASE_DIR, 'scene_usd')
MESH_DIR  = os.path.join(SCENE_DIR, 'meshes')
os.makedirs(MESH_DIR, exist_ok=True)

# ── Coordinate system ─────────────────────────────────────────────────────────
# USD scenes are usually authored in a local Cartesian frame (metres) already,
# with no inherent GPS anchor -- unlike OSM, which is always WGS84-derived.
# If your USD stage carries real-world georeference metadata, set
# USD_HAS_GEOREFERENCE=True and fill in the anchor; otherwise the scene origin
# is just wherever the USD stage's own (0,0,0) is, and you must align it to
# your TX/RX GPS positions manually (e.g. by placing TX at a known USD prim).
USD_HAS_GEOREFERENCE = _env_bool('USD_HAS_GEOREFERENCE', True)     # True -> DEM terrain drape (CELL 2b) is active by default
ANCHOR_LON = -0.13399    # only used if USD_HAS_GEOREFERENCE=True
ANCHOR_LAT =  51.5305

# Fixed scene bbox (WGS84) -- same area as sionna019_scene_builder_london.ipynb's
# SCENE_WEST/EAST/SOUTH/NORTH, so the DEM terrain and OSM clutter (CELL 2b/2c) cover
# the same real-world extent as that notebook, instead of just hugging the USD buildings.
USE_FIXED_SCENE_BBOX = _env_bool('USE_FIXED_SCENE_BBOX', True)
SCENE_WEST  = -0.231017
SCENE_EAST  = -0.036963
SCENE_SOUTH = 51.47014
SCENE_NORTH = 51.59086

# ── Real OSM buildings (when you have no real USD file yet) ────────────────
# Same building source/approach as sionna019_scene_builder_london.ipynb's CELL 4:
# downloads real OSM building footprints + heights + materials and extrudes
# them into usd_meshes, in place of the 3 synthetic demo boxes. Still produces
# the exact same usd_meshes shape, so CELL 3/3b/4/5/6 need no changes.
# Requires USD_HAS_GEOREFERENCE=True (and ideally USE_FIXED_SCENE_BBOX=True).
USE_OVERTURE_BUILDINGS = _env_bool('USE_OVERTURE_BUILDINGS', True)   # Overture Maps (better than OSM, free, no key)
USE_OSM_BUILDINGS      = _env_bool('USE_OSM_BUILDINGS', False)        # fallback if Overture is unavailable
BUILDING_BBOX_RADIUS_M = 500.0   # fallback query radius around ANCHOR_LON/LAT
OVERTURE_RELEASE       = '2026-06-17.0'  # Overture Maps release date; update to latest
                                  # when USE_FIXED_SCENE_BBOX=False
HEIGHT_PER_LEVEL_M   = 3.5        # m per building:levels= storey, when no height= tag
DEFAULT_HEIGHT_M     = 9.0        # fallback height (m) when OSM has neither tag
CITY_MIN_HEIGHT_M    = 3.0
CITY_MAX_HEIGHT_M    = 250.0

# ── Export the built scene as a real .usd file (so it round-trips into
# Omniverse/USD tooling, not just PLY/Mitsuba XML) ─────────────────────────
EXPORT_USD       = _env_bool('EXPORT_USD', True)
EXPORT_USD_FILE  = os.path.join(SCENE_DIR, 'scene_built.usda')

# ── Roof shape (USE_OSM_BUILDINGS) ──────────────────────────────────────────
# Pyramidal-roof ridge height per building, same priority chain as
# sionna019_scene_builder_london.ipynb's _roof_height: roof:shape=flat -> flat;
# roof:height tag; roof:levels tag; else derive from ROOF_PITCH_DEG and the
# footprint's short dimension. Apex-fan roof geometry (not real hip/gable
# facets) -- an approximation, same simplification level as the rest of this
# notebook's extrusion helpers.
ROOF_PITCH_DEG    = 30.0
ROOF_MAX_RIDGE_M  = 15.0

# ── nDSM height fallback for untagged buildings ─────────────────────────────
# When a building has neither height= nor building:levels=, use real LiDAR
# data instead of a flat DEFAULT_HEIGHT_M: nDSM = DSM (surface, incl. roofs) -
# DTM (bare earth) at the building's centroid, same EA WCS service as CELL 2b's
# terrain (auto-discovers the DSM CoverageId the same way). Off by default --
# adds an extra WCS download; needs network access.
USE_NDSM_HEIGHT_FALLBACK = _env_bool('USE_NDSM_HEIGHT_FALLBACK', False)
EA_DSM_TIFF = os.path.join(BASE_DIR, 'dsm.tif')

# ── NVIDIA NIM vision-language material classification (optional) ──────────
# Overrides CELL 3's filename-keyword material guess with an actual visual
# classification of each building's real-world facade, using an NVIDIA NIM
# (build.nvidia.com) hosted vision-language model. Off by default -- needs
# an NVIDIA_API_KEY (https://build.nvidia.com -> API key) and network access.
USE_NVIDIA_MATERIAL_CLASSIFICATION = _env_bool('USE_NVIDIA_MATERIAL_CLASSIFICATION', False)
NVIDIA_API_KEY      = os.environ.get('NVIDIA_API_KEY', '')
NVIDIA_NIM_BASE_URL = 'https://integrate.api.nvidia.com/v1'
NVIDIA_NIM_VLM_MODEL = 'meta/llama-3.2-90b-vision-instruct'  # NIM catalog VLM
# Public, keyless satellite imagery used as the VLM's input crop per building
# (Esri World Imagery REST export) -- swap for street-view imagery for better
# facade-level classification if you have a provider/key for that.
SATELLITE_TILE_URL = ('https://server.arcgisonline.com/ArcGIS/rest/services/'
                      'World_Imagery/MapServer/export')
# Street-level facade imagery (Mapillary, free tier) -- satellite only shows
# the *roof*, which misclassifies facade material (e.g. a glass tower with a
# flat grey roof reads as concrete). Get a free token at mapillary.com/dashboard
# -> Developers -> Register application -> Client token.
MAPILLARY_TOKEN = os.environ.get('MAPILLARY_TOKEN', '')
MATERIAL_IMAGE_SOURCE = os.environ.get('MATERIAL_IMAGE_SOURCE', 'streetview')  # 'streetview' | 'satellite'

# ── Real elevation (EA LiDAR DTM) — drape USD buildings onto real terrain ──
# Requires USD_HAS_GEOREFERENCE=True (ANCHOR_LON/ANCHOR_LAT must be the real
# WGS84 location of the USD stage's local origin (0,0,0)) -- otherwise there
# is no way to look up a real-world elevation for the USD scene's (x, y).
USE_DEM_TERRAIN  = _env_bool('USE_DEM_TERRAIN', True)     # False -> flat terrain (old behaviour)
TERRAIN_GRID_N   = 200      # terrain.ply resolution (N x N grid)
TERRAIN_PAD_M    = 200.0    # terrain extent beyond the USD scene bbox (m)
EA_DTM_TIFF      = os.path.join(BASE_DIR, 'dem.tif')

# ── OSM real-world clutter (roads, vegetation, water) around the USD buildings ──
# USD scenes typically only contain the buildings/objects the artist modeled --
# this fills the gaps with real OSM features for the same bbox, draped onto the
# same DEM terrain as the buildings (CELL 2b). Requires USD_HAS_GEOREFERENCE=True
# (needs ANCHOR_LON/ANCHOR_LAT to know where on Earth the USD scene actually is).
INCLUDE_OSM_ROADS      = _env_bool('INCLUDE_OSM_ROADS', True)
INCLUDE_OSM_VEGETATION = _env_bool('INCLUDE_OSM_VEGETATION', True)
INCLUDE_OSM_WATER      = _env_bool('INCLUDE_OSM_WATER', True)
OSM_BBOX_PAD_M    = 100.0   # extra margin (m) around the USD scene bbox for the OSM query
ROAD_HEIGHT_M     = 0.05    # road mesh thickness
VEGETATION_HEIGHT_M = 8.0   # default tree/canopy height (m) when OSM has no height tag
WATER_HEIGHT_M    = 0.10    # water mesh thickness

# Must match PROJECTION_CRS in sionna2_915mhz_dem_simulation_london.ipynb /
# sionna019_scene_builder_london.ipynb / sionna018_neural_calibration_london.ipynb
PROJECTION_CRS  = 'bng'    # 'bng' | 'utm30n'
_PROJECTION_EPSG_MAP = {'bng': 27700, 'utm30n': 32630}
UTM_EPSG = _PROJECTION_EPSG_MAP.get(PROJECTION_CRS, 27700)

FREQUENCY_HZ = 915.95e6

print(f'Scenario   : {SCENARIO_NAME}')
print(f'Demo mode  : {DEMO_MODE}')
print(f'USD file   : {USD_FILE if not DEMO_MODE else "(none -- synthetic demo scene)"}')
print(f'Scene dir  : {SCENE_DIR}')
print(f'Projection : {PROJECTION_CRS}  ->  EPSG:{UTM_EPSG}')
print(f'DEM terrain: {USE_DEM_TERRAIN and USD_HAS_GEOREFERENCE} ' + ('' if USD_HAS_GEOREFERENCE else '(needs USD_HAS_GEOREFERENCE=True)'))


## CELL 1 — Imports

In [ ]:
# ============================================================
# CELL 1 — IMPORTS & DEPENDENCIES
# ============================================================
import os, math, json
import numpy as np

try:
    import trimesh
    _HAS_TRIMESH = True
except ImportError:
    _HAS_TRIMESH = False
print(f'trimesh  : {"OK" if _HAS_TRIMESH else "not available -- falls back to ASCII PLY writer"}')

# pxr (usd-core) is only needed when loading a real USD file
# (DEMO_MODE=False AND USE_OVERTURE_BUILDINGS=False AND USE_OSM_BUILDINGS=False).
# When USE_OVERTURE_BUILDINGS=True or USE_OSM_BUILDINGS=True, pxr is never called.
_need_usd = (not DEMO_MODE
             and not globals().get('USE_OVERTURE_BUILDINGS', True)
             and not globals().get('USE_OSM_BUILDINGS', False))

_HAS_USD = False
if _need_usd:
    try:
        from pxr import Usd, UsdGeom, UsdShade, Gf
        _HAS_USD = True
        print('pxr (usd-core) : OK')
    except ImportError as _e:
        print(f'pxr (usd-core) : NOT AVAILABLE -- {_e}')
        print('  Install with:  pip install usd-core')
        print('  (Only needed when loading a real .usd file -- not required for Overture/OSM mode)')
else:
    _reason = 'USE_OVERTURE_BUILDINGS=True' if globals().get('USE_OVERTURE_BUILDINGS', True) else (
              'USE_OSM_BUILDINGS=True' if globals().get('USE_OSM_BUILDINGS', False) else 'DEMO_MODE=True')
    print(f'pxr (usd-core) : skipped ({_reason} -- no USD file needed)')


## CELL 2 — Load Building Geometry

**Priority order:**
1. **Overture Maps** (`USE_OVERTURE_BUILDINGS=True`, default) — free, no API key,
   direct S3 anonymous read, 215K buildings for London, backed by Amazon/Meta/Microsoft/TomTom
2. **OSM fallback** (`USE_OSM_BUILDINGS=True`) — OpenStreetMap via Overpass API
3. **USD stage** (`DEMO_MODE=False`, `USD_FILE=...`) — real photogrammetric mesh from Omniverse
4. **Demo** (`DEMO_MODE=True`) — 3 synthetic boxes, no downloads needed

Output: `usd_meshes` list fed into CELL 2b (terrain drape) → CELL 3 (material mapping) → CELL 4 (PLYs).

In [ ]:
# ============================================================
# CELL 2 — LOAD BUILDING GEOMETRY
# ============================================================
# Priority (controlled by CELL 0 flags):
#   1. USE_OVERTURE_BUILDINGS  -> Overture Maps (S3, free, no key, best accuracy)
#   2. USE_OSM_BUILDINGS       -> OpenStreetMap (Overpass API, free, fallback)
#   3. DEMO_MODE / USD_FILE    -> synthetic demo scene or real USD stage
# Output: usd_meshes = [{'name', 'verts', 'faces', 'material', ...}, ...]
# ============================================================
import os, math
import numpy as np

# ── Shared extrusion helpers ─────────────────────────────────────────────────
def _box_mesh(cx, cy, cz, sx, sy, sz):
    x0, x1 = cx - sx/2, cx + sx/2
    y0, y1 = cy - sy/2, cy + sy/2
    z0, z1 = cz, cz + sz
    v = np.array([[x0,y0,z0],[x1,y0,z0],[x1,y1,z0],[x0,y1,z0],
                  [x0,y0,z1],[x1,y0,z1],[x1,y1,z1],[x0,y1,z1]], dtype=np.float32)
    f = np.array([[0,1,2],[0,2,3],[4,6,5],[4,7,6],
                  [0,4,5],[0,5,1],[1,5,6],[1,6,2],
                  [2,6,7],[2,7,3],[3,7,4],[3,4,0]], dtype=np.int32)
    return v, f

def _extrude_footprint(coords_local, wall_height, ridge_height=0.0):
    ring = np.asarray(coords_local, dtype=np.float64)
    if len(ring) >= 2 and np.allclose(ring[0], ring[-1]):
        ring = ring[:-1]
    m = len(ring)
    if m < 3:
        return None, None
    cx, cy = ring[:, 0].mean(), ring[:, 1].mean()
    bottom = np.column_stack([ring, np.zeros(m)])
    eave   = np.column_stack([ring, np.full(m, wall_height)])
    cb, apex = np.array([[cx, cy, 0.0]]), np.array([[cx, cy, wall_height + ridge_height]])
    V = np.vstack([bottom, eave, cb, apex]).astype(np.float32)
    cb_idx, apex_idx = 2 * m, 2 * m + 1
    F = []
    for i in range(m):
        j = (i + 1) % m
        F.append([cb_idx, j, i])
        F.append([apex_idx, m + i, m + j])
        F.append([i, j, m + i])
        F.append([j, m + j, m + i])
    return V, np.asarray(F, dtype=np.int32)

usd_meshes = []

# ── Coordinate transformer (anchor -> local) ──────────────────────────────
from pyproj import Transformer
_wgs_to_proj = Transformer.from_crs('EPSG:4326', f'EPSG:{UTM_EPSG}', always_xy=True)
_anchor_e, _anchor_n = _wgs_to_proj.transform(ANCHOR_LON, ANCHOR_LAT)

def _to_local(lon, lat):
    e, n = _wgs_to_proj.transform(lon, lat)
    return e - _anchor_e, n - _anchor_n

if globals().get('USE_FIXED_SCENE_BBOX', False):
    _bw, _bs, _be, _bn = SCENE_WEST, SCENE_SOUTH, SCENE_EAST, SCENE_NORTH
else:
    _dlat = BUILDING_BBOX_RADIUS_M / 111320.0
    _dlon = BUILDING_BBOX_RADIUS_M / (111320.0 * math.cos(math.radians(ANCHOR_LAT)))
    _bw, _be = ANCHOR_LON - _dlon, ANCHOR_LON + _dlon
    _bs, _bn = ANCHOR_LAT - _dlat, ANCHOR_LAT + _dlat

# ── Height / material helpers ────────────────────────────────────────────────
_MAT_OVERTURE_MAP = {
    'concrete': 'Concrete', 'reinforced_concrete': 'Concrete', 'cement': 'Concrete',
    'brick': 'Brick', 'stone': 'Brick', 'sandstone': 'Brick', 'limestone': 'Brick',
    'wood': 'Wood', 'timber': 'Wood',
    'glass': 'Glass', 'glazed': 'Glass',
    'metal': 'Metal', 'steel': 'Metal', 'aluminium': 'Metal', 'aluminum': 'Metal',
}

def _mat_from_str(s):
    if not s:
        return None
    sl = str(s).lower()
    for k, v in _MAT_OVERTURE_MAP.items():
        if k in sl:
            return v
    return None

def _bld_height_overture(height_val, num_floors_val):
    try:
        h = float(height_val)
        if h > 1:
            return float(np.clip(h, CITY_MIN_HEIGHT_M, CITY_MAX_HEIGHT_M)), False
    except (TypeError, ValueError):
        pass
    try:
        lvl = float(num_floors_val)
        if lvl > 0:
            return float(np.clip(lvl * HEIGHT_PER_LEVEL_M, CITY_MIN_HEIGHT_M, CITY_MAX_HEIGHT_M)), False
    except (TypeError, ValueError):
        pass
    return None, True

def _ridge_height_overture(roof_shape, roof_height_val, coords_local):
    shape = str(roof_shape or '').lower().strip()
    if shape in ('flat', 'terrace'):
        return 0.0
    try:
        rh = float(roof_height_val)
        if rh > 0.2:
            return float(np.clip(rh, 0.0, ROOF_MAX_RIDGE_M))
    except (TypeError, ValueError):
        pass
    try:
        pa = np.asarray(coords_local, dtype=float)
        span = min(pa[:, 0].max() - pa[:, 0].min(), pa[:, 1].max() - pa[:, 1].min())
        return float(np.clip((span / 2.0) * math.tan(math.radians(ROOF_PITCH_DEG)), 0.0, ROOF_MAX_RIDGE_M))
    except Exception:
        return 0.0

# ============================================================
# PATH A — OVERTURE MAPS  (free, no API key, S3 direct read)
# ============================================================
_overture_ok = False
if globals().get('USE_OVERTURE_BUILDINGS', True) and USD_HAS_GEOREFERENCE:
    print('Building from Overture Maps (S3 direct, anonymous)...')
    try:
        import pyarrow.dataset as _pad
        import pyarrow.fs as _pafs
        import shapely.wkb as _swkb
        import shapely.geometry as _sg
        import time as _time

        _s3 = _pafs.S3FileSystem(region='us-west-2', anonymous=True)
        _release = globals().get('OVERTURE_RELEASE', '2026-06-17.0')
        _base = f'overturemaps-us-west-2/release/{_release}/theme=buildings/type=building/'

        _t0 = _time.time()
        _ds = _pad.dataset(_base, filesystem=_s3, format='parquet')
        _filt = (
            (_pad.field('bbox', 'xmin') < _be) & (_pad.field('bbox', 'xmax') > _bw) &
            (_pad.field('bbox', 'ymin') < _bn) & (_pad.field('bbox', 'ymax') > _bs)
        )
        _cols = ['height', 'num_floors', 'facade_material', 'roof_shape', 'roof_height', 'geometry']
        _tbl = _ds.to_table(filter=_filt, columns=_cols)
        print(f'  {len(_tbl)} buildings downloaded in {_time.time()-_t0:.1f}s  '
              f'(release {_release})')
        print(f'  Height available: {len(_tbl) - _tbl["height"].null_count} / {len(_tbl)}  '
              f'  Floors: {len(_tbl) - _tbl["num_floors"].null_count} / {len(_tbl)}')

        _n_bld = 0
        for _i in range(len(_tbl)):
            _geom_raw = _tbl['geometry'][_i].as_py()
            if not _geom_raw:
                continue
            try:
                _geom = _swkb.loads(bytes(_geom_raw))
            except Exception:
                continue
            if _geom is None or _geom.is_empty:
                continue

            _polys = list(_geom.geoms) if isinstance(_geom, _sg.MultiPolygon) else [_geom]
            _h_val  = _tbl['height'][_i].as_py()
            _nf_val = _tbl['num_floors'][_i].as_py()
            _fm_val = _tbl['facade_material'][_i].as_py()
            _rs_val = _tbl['roof_shape'][_i].as_py()
            _rh_val = _tbl['roof_height'][_i].as_py()

            height, _is_default = _bld_height_overture(_h_val, _nf_val)
            if _is_default:
                height = DEFAULT_HEIGHT_M
            material = _mat_from_str(_fm_val) or 'Concrete'

            for _p in _polys:
                if _p.is_empty or _p.area <= 0:
                    continue
                coords_local = [_to_local(lon, lat) for lon, lat in _p.exterior.coords]
                ridge = _ridge_height_overture(_rs_val, _rh_val, coords_local)
                V, F = _extrude_footprint(coords_local, height, ridge)
                if V is None:
                    continue
                usd_meshes.append({
                    'name': f'ov_building_{_i}',
                    'verts': V, 'faces': F,
                    'material': material,
                    '_height_is_default': _is_default,
                    '_orig_height': height,
                })
                _n_bld += 1

        # Flat ground slab (replaced by real DEM terrain in CELL 2b)
        if usd_meshes:
            _all_xy = np.concatenate([m['verts'][:, :2] for m in usd_meshes], axis=0)
            _gcx = (_all_xy[:, 0].max() + _all_xy[:, 0].min()) / 2.0
            _gcy = (_all_xy[:, 1].max() + _all_xy[:, 1].min()) / 2.0
            _gx  = (_all_xy[:, 0].max() - _all_xy[:, 0].min()) + 2 * BUILDING_BBOX_RADIUS_M
            _gy  = (_all_xy[:, 1].max() - _all_xy[:, 1].min()) + 2 * BUILDING_BBOX_RADIUS_M
            v, f = _box_mesh(_gcx, _gcy, 0, _gx, _gy, 0.2)
            usd_meshes.append({'name': 'ground', 'verts': v, 'faces': f, 'material': 'Concrete_Ground'})

        print(f'  {_n_bld} building mesh(es) extruded from Overture Maps footprints')
        _overture_ok = bool(usd_meshes)

    except Exception as _e:
        print(f'  Overture Maps failed: {_e}')
        print('  Falling back to OSM ...')
        usd_meshes = []

# ============================================================
# PATH B — OSM FALLBACK  (if Overture unavailable / disabled)
# ============================================================
_osm_buildings_ok = False
if not _overture_ok and globals().get('USE_OSM_BUILDINGS', False) and USD_HAS_GEOREFERENCE:
    print('Building from OSM (Overpass API fallback)...')
    try:
        import osmnx as ox
        import shapely.geometry as sg

        ox.settings.use_cache = True
        ox.settings.log_console = False
        ox.settings.requests_timeout = 30

        _ox_version = tuple(int(x) for x in ox.__version__.split('.')[:2])
        if _ox_version >= (2, 0):
            gdf_bld = ox.features_from_bbox(bbox=(_bw, _bs, _be, _bn), tags={'building': True})
        elif _ox_version >= (1, 3):
            gdf_bld = ox.features_from_bbox(bbox=(_bn, _bs, _be, _bw), tags={'building': True})
        else:
            gdf_bld = ox.features_from_bbox(north=_bn, south=_bs, east=_be, west=_bw, tags={'building': True})
        print(f'  {len(gdf_bld)} raw OSM building feature(s)')

        _MAT_TAG_MAP = {
            'concrete': 'Concrete', 'reinforced_concrete': 'Concrete',
            'brick': 'Brick', 'stone': 'Brick',
            'glass': 'Glass', 'metal': 'Metal', 'steel': 'Metal',
            'wood': 'Wood', 'timber': 'Wood',
        }
        def _osm_bld_material(row):
            for _tag in ('building:material', 'building:facade:material'):
                _s = str(row.get(_tag, '')).lower()
                for k, v in _MAT_TAG_MAP.items():
                    if k in _s:
                        return v
            return 'Concrete'

        def _osm_bld_height(row):
            try:
                h = float(str(row.get('height', '0')).replace('m','').strip())
                if h > 1: return float(np.clip(h, CITY_MIN_HEIGHT_M, CITY_MAX_HEIGHT_M)), False
            except Exception: pass
            try:
                lvl = float(str(row.get('building:levels', '0')).strip())
                if lvl > 0: return float(np.clip(lvl * HEIGHT_PER_LEVEL_M, CITY_MIN_HEIGHT_M, CITY_MAX_HEIGHT_M)), False
            except Exception: pass
            return None, True

        _n_osm = 0
        for _i, (_, row) in enumerate(gdf_bld.iterrows()):
            geom = row.geometry
            if geom is None or geom.is_empty: continue
            polys = list(geom.geoms) if isinstance(geom, sg.MultiPolygon) else [geom]
            height, _is_def = _osm_bld_height(row)
            if _is_def: height = DEFAULT_HEIGHT_M
            material = _osm_bld_material(row)
            for _p in polys:
                if _p.is_empty or _p.area <= 0: continue
                coords_local = [_to_local(lon, lat) for lon, lat in _p.exterior.coords]
                ridge = _ridge_height_overture(row.get('roof:shape'), row.get('roof:height'), coords_local)
                V, F = _extrude_footprint(coords_local, height, ridge)
                if V is None: continue
                usd_meshes.append({'name': f'osm_building_{_i}', 'verts': V, 'faces': F,
                                   'material': material, '_height_is_default': _is_def, '_orig_height': height})
                _n_osm += 1

        if usd_meshes:
            _all_xy = np.concatenate([m['verts'][:, :2] for m in usd_meshes], axis=0)
            _gcx = (_all_xy[:, 0].max() + _all_xy[:, 0].min()) / 2.0
            _gcy = (_all_xy[:, 1].max() + _all_xy[:, 1].min()) / 2.0
            _gx  = (_all_xy[:, 0].max() - _all_xy[:, 0].min()) + 2 * BUILDING_BBOX_RADIUS_M
            _gy  = (_all_xy[:, 1].max() - _all_xy[:, 1].min()) + 2 * BUILDING_BBOX_RADIUS_M
            v, f = _box_mesh(_gcx, _gcy, 0, _gx, _gy, 0.2)
            usd_meshes.append({'name': 'ground', 'verts': v, 'faces': f, 'material': 'Concrete_Ground'})
        print(f'  {_n_osm} building mesh(es) from OSM')
        _osm_buildings_ok = bool(usd_meshes)

    except Exception as _e:
        print(f'  OSM download failed: {_e}')
        usd_meshes = []

# ============================================================
# PATH C — USD STAGE (real file) or DEMO (synthetic)
# ============================================================
if not _overture_ok and not _osm_buildings_ok:
    if DEMO_MODE or not _HAS_USD:
        print('Building synthetic demo scene (3 boxes + ground) ...')
        v, f = _box_mesh(0, 0, 0, 400, 300, 0.2);   usd_meshes.append({'name': 'ground',     'verts': v, 'faces': f, 'material': 'Concrete_Ground'})
        v, f = _box_mesh(-80, 60, 0, 40, 30, 22.0); usd_meshes.append({'name': 'building_A', 'verts': v, 'faces': f, 'material': 'Brick_Facade'})
        v, f = _box_mesh(60, -40, 0, 50, 50, 35.0); usd_meshes.append({'name': 'building_B', 'verts': v, 'faces': f, 'material': 'Glass_Curtain_Wall'})
        v, f = _box_mesh(20, 90, 0, 30, 20, 12.0);  usd_meshes.append({'name': 'building_C', 'verts': v, 'faces': f, 'material': 'Metal_Cladding'})
        print(f'  {len(usd_meshes)} synthetic prims (demo)')
    else:
        print(f'Opening USD stage: {USD_FILE}')
        from pxr import Usd, UsdGeom, UsdShade, Gf
        stage = Usd.Stage.Open(USD_FILE)
        assert stage, f'Could not open: {USD_FILE}'
        xform_cache = UsdGeom.XformCache()

        def _triangulate(counts, idxs):
            tris, idx = [], 0
            for n in counts:
                poly = idxs[idx: idx + n]
                for k in range(1, n - 1):
                    tris.append([poly[0], poly[k], poly[k + 1]])
                idx += n
            return np.asarray(tris, dtype=np.int32)

        def _bound_mat(prim):
            try:
                rel = UsdShade.MaterialBindingAPI(prim).ComputeBoundMaterial()
                mat = rel[0] if isinstance(rel, tuple) else rel
                if mat and mat.GetPrim().IsValid():
                    return mat.GetPrim().GetName()
            except Exception: pass
            return 'DEFAULT'

        for prim in stage.Traverse():
            if not prim.IsA(UsdGeom.Mesh): continue
            mesh = UsdGeom.Mesh(prim)
            pts = np.asarray(mesh.GetPointsAttr().Get(), dtype=np.float64)
            if pts is None or len(pts) == 0: continue
            faces = _triangulate(mesh.GetFaceVertexCountsAttr().Get(), mesh.GetFaceVertexIndicesAttr().Get())
            world = xform_cache.GetLocalToWorldTransform(prim)
            pts_w = (np.hstack([pts, np.ones((len(pts), 1))]) @ np.array(world).reshape(4,4))[:, :3].astype(np.float32)
            usd_meshes.append({'name': prim.GetName(), 'verts': pts_w, 'faces': faces, 'material': _bound_mat(prim)})
        print(f'  {len(usd_meshes)} mesh prims from USD stage')

print(f'\nTotal meshes in usd_meshes: {len(usd_meshes)}')
for m in usd_meshes[:5]:
    print(f"  {m['name']:<20} verts={len(m['verts']):>5}  faces={len(m['faces']):>5}  material={m['material']}")
if len(usd_meshes) > 5:
    print(f'  ... and {len(usd_meshes)-5} more')


## CELL 2b — EA LiDAR DTM Terrain + Drape USD Buildings onto Real Elevation
Downloads the Environment Agency 1 m Composite DTM (bare-earth terrain) for
the bbox around `(ANCHOR_LON, ANCHOR_LAT)`, builds a real `terrain.ply` from
it (same grid-sampling approach as `sionna019_scene_builder_london.ipynb`
CELL 3), then **drapes** every USD mesh by shifting its vertices' Z by the
real terrain height at its footprint centroid — since USD meshes are
typically authored on a local flat (z=0) ground plane, not on real elevation.

Skips automatically (flat terrain, old behaviour) if `USE_DEM_TERRAIN=False`
or `USD_HAS_GEOREFERENCE=False` (no real-world anchor to look up elevation
against).


In [ ]:
# ============================================================
# CELL 2b — EA LIDAR DTM TERRAIN + DRAPE
# ============================================================
_dem_ok = False

if not (USE_DEM_TERRAIN and USD_HAS_GEOREFERENCE):
    print('Skipping DEM terrain -- USE_DEM_TERRAIN/USD_HAS_GEOREFERENCE not both True.')
    print('Buildings stay on the flat z=0 plane they were authored on.')
else:
    import requests
    import numpy as np
    from pyproj import Transformer

    print('=' * 60)
    print('CELL 2b -- EA LiDAR DTM download + terrain drape')
    print('=' * 60)

    # USD local (x, y) in metres around the anchor -> real-world BNG/UTM (e, n)
    _wgs_to_bng = Transformer.from_crs('EPSG:4326', f'EPSG:{UTM_EPSG}', always_xy=True)
    _bng_to_wgs = Transformer.from_crs(f'EPSG:{UTM_EPSG}', 'EPSG:4326', always_xy=True)
    _anchor_e, _anchor_n = _wgs_to_bng.transform(ANCHOR_LON, ANCHOR_LAT)

    # Scene extent in USD local metres, from the meshes already loaded in CELL 2
    _all_xy = np.concatenate([m['verts'][:, :2] for m in usd_meshes], axis=0)
    _x_min, _x_max = float(_all_xy[:, 0].min()), float(_all_xy[:, 0].max())
    _y_min, _y_max = float(_all_xy[:, 1].min()), float(_all_xy[:, 1].max())

    if globals().get('USE_FIXED_SCENE_BBOX', False):
        # Same fixed area as sionna019_scene_builder_london.ipynb's SCENE_WEST/
        # EAST/SOUTH/NORTH, converted to BNG, instead of just padding the USD bbox.
        _e_min, _n_min = _wgs_to_bng.transform(SCENE_WEST, SCENE_SOUTH)
        _e_max, _n_max = _wgs_to_bng.transform(SCENE_EAST, SCENE_NORTH)
    else:
        _e_min = _anchor_e + _x_min - TERRAIN_PAD_M
        _e_max = _anchor_e + _x_max + TERRAIN_PAD_M
        _n_min = _anchor_n + _y_min - TERRAIN_PAD_M
        _n_max = _anchor_n + _y_max + TERRAIN_PAD_M
    print(f'Scene extent (local) : x=[{_x_min:.0f},{_x_max:.0f}]  y=[{_y_min:.0f},{_y_max:.0f}]')
    print(f'Real-world bbox (EPSG:{UTM_EPSG}) : E=[{_e_min:.0f},{_e_max:.0f}]  N=[{_n_min:.0f},{_n_max:.0f}]')

    _WCS_ENDPOINT = 'https://environment.data.gov.uk/spatialdata/lidar-composite-digital-terrain-model-dtm-1m/wcs'

    def _wcs_get_coverage_id(default='13787b9a-26a4-4775-8523-806d13af58fc__Lidar_Composite_Elevation_DTM_1m'):
        """Auto-detect the real CoverageId via GetCapabilities -- the EA WCS
        has renamed/changed this id before, which surfaces as a 404 or a
        500 (server throws on an unrecognised id) on GetCoverage even
        though the service itself is up."""
        try:
            _cap = requests.get(_WCS_ENDPOINT, params={
                'SERVICE': 'WCS', 'VERSION': '2.0.1', 'REQUEST': 'GetCapabilities'},
                timeout=30)
            _cap.raise_for_status()
            import re as _re
            _ids = _re.findall(r'<(?:wcs:)?CoverageId>([^<]+)</(?:wcs:)?CoverageId>', _cap.text)
            print(f'  GetCapabilities advertises {len(_ids)} coverage(s): {_ids[:10]}'
                  f'{" ..." if len(_ids) > 10 else ""}')
            if _ids:
                _dtm_ids = [i for i in _ids if 'dtm' in i.lower()] or _ids
                if default not in _dtm_ids:
                    print(f'  CoverageId "{default}" not advertised; '
                          f'using "{_dtm_ids[0]}" instead.')
                return _dtm_ids[0] if default not in _dtm_ids else default
        except Exception as _e:
            print(f'  GetCapabilities probe failed: {_e} -- keeping default CoverageId.')
        return default

    if os.path.exists(EA_DTM_TIFF):
        print(f'Already downloaded: {EA_DTM_TIFF}')
    else:
        def _wcs_url(coverage_id):
            return (
                _WCS_ENDPOINT +
                '?SERVICE=WCS&VERSION=2.0.1&REQUEST=GetCoverage'
                f'&COVERAGEID={coverage_id}'
                f'&SUBSET=E,http://www.opengis.net/def/crs/EPSG/0/27700({_e_min:.0f},{_e_max:.0f})'
                f'&SUBSET=N,http://www.opengis.net/def/crs/EPSG/0/27700({_n_min:.0f},{_n_max:.0f})'
                '&FORMAT=image/tiff'
            )

        print('Downloading EA LiDAR DTM ...')
        _coverage_ids_to_try = ['13787b9a-26a4-4775-8523-806d13af58fc__Lidar_Composite_Elevation_DTM_1m']
        try:
            _r = requests.get(_wcs_url(_coverage_ids_to_try[0]), timeout=120)
            if _r.status_code in (404, 500):
                # CoverageId mismatch is the most common cause of a 404/500 on
                # an otherwise-valid, small, in-coverage bbox (a 500 means the
                # server itself threw on an unrecognised id rather than
                # cleanly 404ing) -- re-probe via GetCapabilities and retry
                # once with the discovered id.
                print(f'  {_r.status_code} on CoverageId "{_coverage_ids_to_try[0]}" -- probing GetCapabilities ...')
                _alt_id = _wcs_get_coverage_id(_coverage_ids_to_try[0])
                if _alt_id != _coverage_ids_to_try[0]:
                    _coverage_ids_to_try.append(_alt_id)
                    _r = requests.get(_wcs_url(_alt_id), timeout=120)
            _r.raise_for_status()
            os.makedirs(os.path.dirname(EA_DTM_TIFF), exist_ok=True)
            with open(EA_DTM_TIFF, 'wb') as _f:
                _f.write(_r.content)
            print(f'  saved {os.path.getsize(EA_DTM_TIFF) // 1024} KB -> {EA_DTM_TIFF}')
        except Exception as _e:
            print(f'  WCS download failed: {_e}')
            print('  Tried CoverageId(s): ' + ', '.join(_coverage_ids_to_try))
            print('  If this persists, the WCS service may be down or your bbox is offshore/'
                  'outside England. Manual fallback: download the 1m DTM tile for your area from')
            print('  https://environment.data.gov.uk/DefraDataDownload/?Mode=survey '
                  '(LIDAR Composite DTM, 1m), then point EA_DTM_TIFF (CELL 0) at the .tif directly.')

    if os.path.exists(EA_DTM_TIFF):
        try:
            import rasterio
            with rasterio.open(EA_DTM_TIFF) as _ds:
                _dem_arr = _ds.read(1).astype(np.float32)
                _dem_transform = _ds.transform
            _rows, _cols = _dem_arr.shape
            _dem_es = np.array([_dem_transform * (c + 0.5, 0) for c in range(_cols)])[:, 0]
            _dem_ns = np.array([_dem_transform * (0, r + 0.5) for r in range(_rows)])[:, 1]

            from scipy.interpolate import RegularGridInterpolator
            _dem_interp = RegularGridInterpolator(
                (_dem_ns[::-1], _dem_es), _dem_arr[::-1, :],
                bounds_error=False, fill_value=None)

            def local_z(x, y):
                """Real terrain height (m) at USD local (x, y), relative to anchor elevation."""
                e, n = _anchor_e + x, _anchor_n + y
                return float(_dem_interp([[n, e]])[0]) - _origin_elev_asl

            _origin_elev_asl = float(_dem_interp([[_anchor_n, _anchor_e]])[0])
            print(f'Anchor elevation : {_origin_elev_asl:.2f} m ASL')

            # ── nDSM height fallback for untagged OSM buildings (USE_NDSM_HEIGHT_FALLBACK) ──
            # nDSM = DSM (surface, incl. roofs) - DTM (bare earth) at each building's
            # centroid -- rescales any building that fell back to DEFAULT_HEIGHT_M in
            # CELL 2 (no height=/levels= tag) to its real LiDAR-measured height instead.
            if globals().get('USE_NDSM_HEIGHT_FALLBACK', False):
                _default_meshes = [m for m in usd_meshes if m.get('_height_is_default')]
                if not _default_meshes:
                    print('nDSM height fallback: no untagged buildings to rescale.')
                else:
                    try:
                        _dsm_url_base = ('https://environment.data.gov.uk/spatialdata/'
                                          'lidar-composite-digital-surface-model-dsm-1m/wcs')

                        def _dsm_get_coverage_id(default='Lidar_Composite_Elevation_DSM_1m'):
                            try:
                                _cap = requests.get(_dsm_url_base, params={
                                    'SERVICE': 'WCS', 'VERSION': '2.0.1', 'REQUEST': 'GetCapabilities'},
                                    timeout=30)
                                _cap.raise_for_status()
                                import re as _re
                                _ids = _re.findall(r'<(?:wcs:)?CoverageId>([^<]+)</(?:wcs:)?CoverageId>', _cap.text)
                                _dsm_ids = [i for i in _ids if 'dsm' in i.lower()] or _ids
                                return _dsm_ids[0] if _dsm_ids else default
                            except Exception as _e:
                                print(f'  DSM GetCapabilities probe failed: {_e}')
                                return default

                        if not os.path.exists(EA_DSM_TIFF):
                            _dsm_cov_id = _dsm_get_coverage_id()
                            _dsm_url = (
                                _dsm_url_base +
                                '?SERVICE=WCS&VERSION=2.0.1&REQUEST=GetCoverage'
                                f'&COVERAGEID={_dsm_cov_id}'
                                f'&SUBSET=E,http://www.opengis.net/def/crs/EPSG/0/27700({_e_min:.0f},{_e_max:.0f})'
                                f'&SUBSET=N,http://www.opengis.net/def/crs/EPSG/0/27700({_n_min:.0f},{_n_max:.0f})'
                                '&FORMAT=image/tiff'
                            )
                            _dr = requests.get(_dsm_url, timeout=120)
                            _dr.raise_for_status()
                            os.makedirs(os.path.dirname(EA_DSM_TIFF), exist_ok=True)
                            with open(EA_DSM_TIFF, 'wb') as _f:
                                _f.write(_dr.content)
                            print(f'  DSM saved {os.path.getsize(EA_DSM_TIFF) // 1024} KB -> {EA_DSM_TIFF}')

                        with rasterio.open(EA_DSM_TIFF) as _dsm_ds:
                            _dsm_arr = _dsm_ds.read(1).astype(np.float32)
                            _dsm_transform = _dsm_ds.transform
                        _dsm_rows, _dsm_cols = _dsm_arr.shape
                        _dsm_es = np.array([_dsm_transform * (c + 0.5, 0) for c in range(_dsm_cols)])[:, 0]
                        _dsm_ns = np.array([_dsm_transform * (0, r + 0.5) for r in range(_dsm_rows)])[:, 1]
                        _dsm_interp = RegularGridInterpolator(
                            (_dsm_ns[::-1], _dsm_es), _dsm_arr[::-1, :],
                            bounds_error=False, fill_value=None)

                        _n_rescaled = 0
                        for m in _default_meshes:
                            cx, cy = float(m['verts'][:, 0].mean()), float(m['verts'][:, 1].mean())
                            e, n = _anchor_e + cx, _anchor_n + cy
                            ndsm = float(_dsm_interp([[n, e]])[0]) - float(_dem_interp([[n, e]])[0])
                            if ndsm > CITY_MIN_HEIGHT_M:
                                ndsm = float(np.clip(ndsm, CITY_MIN_HEIGHT_M, CITY_MAX_HEIGHT_M))
                                factor = ndsm / m['_orig_height']
                                m['verts'][:, 2] *= factor
                                _n_rescaled += 1
                        print(f'nDSM height fallback: rescaled {_n_rescaled}/{len(_default_meshes)} '
                              f'untagged building(s) to real LiDAR height.')
                    except ImportError as _e:
                        print(f'nDSM height fallback skipped -- rasterio/scipy not available ({_e}).')
                    except Exception as _e:
                        print(f'nDSM height fallback skipped -- {_e}')

            # ── Build real terrain.ply over the scene extent ──────────────
            _xs = np.linspace(_x_min - TERRAIN_PAD_M, _x_max + TERRAIN_PAD_M, TERRAIN_GRID_N, dtype=np.float32)
            _ys = np.linspace(_y_min - TERRAIN_PAD_M, _y_max + TERRAIN_PAD_M, TERRAIN_GRID_N, dtype=np.float32)
            _XX, _YY = np.meshgrid(_xs, _ys)
            _ZZ = np.array([[local_z(float(xv), float(yv)) for xv in _xs] for yv in _ys], dtype=np.float32)
            _tv = np.stack([_XX, _YY, _ZZ], axis=-1).reshape(-1, 3)
            _tf = []
            for _r in range(TERRAIN_GRID_N - 1):
                for _c in range(TERRAIN_GRID_N - 1):
                    _i0 = _r * TERRAIN_GRID_N + _c
                    _i1 = _i0 + 1
                    _i2 = _i0 + TERRAIN_GRID_N
                    _i3 = _i2 + 1
                    _tf.append([_i0, _i2, _i1]); _tf.append([_i1, _i2, _i3])
            _tf = np.asarray(_tf, dtype=np.int32)

            # Replace any synthetic 'ground' mesh with the real DEM terrain mesh
            usd_meshes = [m for m in usd_meshes if m['name'] != 'ground']
            usd_meshes.append({'name': 'dem_terrain', 'verts': _tv, 'faces': _tf,
                                'material': 'EA_LiDAR_Terrain'})

            # ── Drape every building mesh: shift Z by terrain height at its footprint centroid ──
            for m in usd_meshes:
                if m['name'] == 'dem_terrain':
                    continue
                cx, cy = float(m['verts'][:, 0].mean()), float(m['verts'][:, 1].mean())
                dz = local_z(cx, cy)
                m['verts'] = m['verts'].copy()
                m['verts'][:, 2] += dz
                print(f"  draped {m['name']:<16} +{dz:6.2f} m  (terrain height at footprint centroid)")

            _dem_ok = True
            print(f'Terrain grid: {TERRAIN_GRID_N}x{TERRAIN_GRID_N}  '
                  f'z range [{_ZZ.min():.1f}, {_ZZ.max():.1f}] m (relative to anchor)')
        except ImportError as _e:
            print(f'  rasterio/scipy not available ({_e}) -- pip install rasterio scipy')
        except Exception as _e:
            print(f'  DEM processing failed: {_e}')

print(f'DEM terrain applied: {_dem_ok}')


## CELL 2c — OSM Roads, Vegetation, Water (real-world clutter)
USD scenes usually only contain the objects an artist explicitly modeled --
this fills the gaps around the USD buildings with real OpenStreetMap roads,
vegetation, and water for the same bbox (same tag patterns as
`sionna019_scene_builder_london.ipynb` CELL 4), draped onto the same DEM
terrain as the buildings in CELL 2b.

Roads -> buffered by per-highway-type half-width -> `itu_asphalt`.
Vegetation (forest/orchard/wood/scrub) -> extruded to canopy height -> `itu_vegetation`.
Water (natural=water) -> thin extruded polygon -> `itu_water`.

All produced meshes are appended to `usd_meshes`, so CELL 3 (material
mapping) and CELL 4/5 (PLY + scene.xml writers) pick them up automatically
-- no changes needed downstream.

Skips gracefully if `osmnx`/`shapely` aren't installed, or if
`USD_HAS_GEOREFERENCE=False` (no real-world bbox to query OSM against).


In [ ]:
# ============================================================
# CELL 2c — OSM ROADS + VEGETATION + WATER (REAL-WORLD CLUTTER)
# ============================================================
_osm_clutter_added = 0

if not USD_HAS_GEOREFERENCE:
    print('Skipping OSM clutter -- USD_HAS_GEOREFERENCE=False (no real-world bbox).')
else:
    try:
        import osmnx as ox
        ox.settings.use_cache = True
        ox.settings.log_console = False
        ox.settings.requests_timeout = 30  # fail fast instead of hanging on restricted networks
        _HAS_OSMNX = True
    except ImportError as _e:
        _HAS_OSMNX = False
        print(f'osmnx not available ({_e}) -- pip install osmnx. Skipping OSM clutter.')

    if _HAS_OSMNX:
        import numpy as np
        from pyproj import Transformer
        import shapely.geometry as sg

        _wgs_to_bng2 = Transformer.from_crs('EPSG:4326', f'EPSG:{UTM_EPSG}', always_xy=True)
        _bng_to_wgs2 = Transformer.from_crs(f'EPSG:{UTM_EPSG}', 'EPSG:4326', always_xy=True)
        _anchor_e2, _anchor_n2 = _wgs_to_bng2.transform(ANCHOR_LON, ANCHOR_LAT)

        _all_xy2 = np.concatenate([m['verts'][:, :2] for m in usd_meshes
                                    if m['name'] != 'dem_terrain'], axis=0)
        _x0, _x1 = float(_all_xy2[:, 0].min()), float(_all_xy2[:, 0].max())
        _y0, _y1 = float(_all_xy2[:, 1].min()), float(_all_xy2[:, 1].max())
        if globals().get('USE_FIXED_SCENE_BBOX', False):
            # Same fixed area as sionna019_scene_builder_london.ipynb's SCENE_WEST/
            # EAST/SOUTH/NORTH, instead of just padding the USD building footprint.
            _w_lon, _s_lat, _e_lon, _n_lat = SCENE_WEST, SCENE_SOUTH, SCENE_EAST, SCENE_NORTH
        else:
            _e0 = _anchor_e2 + _x0 - OSM_BBOX_PAD_M
            _e1 = _anchor_e2 + _x1 + OSM_BBOX_PAD_M
            _n0 = _anchor_n2 + _y0 - OSM_BBOX_PAD_M
            _n1 = _anchor_n2 + _y1 + OSM_BBOX_PAD_M
            _w_lon, _s_lat = _bng_to_wgs2.transform(_e0, _n0)
            _e_lon, _n_lat = _bng_to_wgs2.transform(_e1, _n1)
        print(f'OSM query bbox: lon=[{_w_lon:.5f},{_e_lon:.5f}]  lat=[{_s_lat:.5f},{_n_lat:.5f}]')

        import osmnx as _ox_ver
        _ox_version = tuple(int(x) for x in _ox_ver.__version__.split('.')[:2])

        def _features_from_bbox(tags):
            if _ox_version >= (2, 0):
                return ox.features_from_bbox(bbox=(_w_lon, _s_lat, _e_lon, _n_lat), tags=tags)
            elif _ox_version >= (1, 3):
                return ox.features_from_bbox(bbox=(_n_lat, _s_lat, _e_lon, _w_lon), tags=tags)
            else:
                return ox.features_from_bbox(north=_n_lat, south=_s_lat, east=_e_lon, west=_w_lon, tags=tags)

        def _to_local(lon, lat):
            e, n = _wgs_to_bng2.transform(lon, lat)
            return e - _anchor_e2, n - _anchor_n2

        def _fan_triangulate_ring(coords2d):
            """Simple centroid-fan triangulation. Exact for convex rings;
            an approximation for concave OSM footprints -- good enough for
            flat radio-material clutter, not a CAD-accurate mesh."""
            coords2d = np.asarray(coords2d, dtype=np.float64)
            if len(coords2d) >= 2 and np.allclose(coords2d[0], coords2d[-1]):
                coords2d = coords2d[:-1]
            return coords2d

        def _extrude_ring(coords2d, z0, height):
            ring = _fan_triangulate_ring(coords2d)
            m = len(ring)
            if m < 3:
                return None, None
            cx, cy = ring[:, 0].mean(), ring[:, 1].mean()
            bottom = np.column_stack([ring, np.full(m, z0)])
            top    = np.column_stack([ring, np.full(m, z0 + height)])
            cb = np.array([[cx, cy, z0]])
            ct = np.array([[cx, cy, z0 + height]])
            V = np.vstack([bottom, top, cb, ct]).astype(np.float32)
            cb_idx, ct_idx = 2 * m, 2 * m + 1
            F = []
            for i in range(m):
                j = (i + 1) % m
                F.append([cb_idx, j, i])                  # bottom fan (down-facing)
                F.append([ct_idx, m + i, m + j])           # top fan (up-facing)
                F.append([i, j, m + i])                     # wall tri 1
                F.append([j, m + j, m + i])                  # wall tri 2
            return V, np.asarray(F, dtype=np.int32)

        def _local_z_safe(x, y):
            if 'local_z' in dir():
                try:
                    return local_z(x, y)
                except Exception:
                    return 0.0
            return 0.0

        def _append_clutter_polygon(poly, height, name_prefix, material, idx):
            if poly.is_empty:
                return 0
            polys = list(poly.geoms) if isinstance(poly, sg.MultiPolygon) else [poly]
            n_added = 0
            for p in polys:
                if p.is_empty or p.area <= 0:
                    continue
                coords_local = [_to_local(lon, lat) for lon, lat in p.exterior.coords]
                V2, F2 = _extrude_ring(coords_local, 0.0, height)
                if V2 is None:
                    continue
                cx, cy = V2[:, 0].mean(), V2[:, 1].mean()
                dz = _local_z_safe(cx, cy)
                V2[:, 2] += dz
                usd_meshes.append({'name': f'{name_prefix}_{idx}_{n_added}',
                                    'verts': V2, 'faces': F2, 'material': material})
                n_added += 1
            return n_added

        # ── Roads ──────────────────────────────────────────────────────────
        if INCLUDE_OSM_ROADS:
            try:
                gdf_roads = _features_from_bbox({'highway': True})
                _ROAD_WIDTH = {'motorway': 12, 'trunk': 10, 'primary': 8, 'secondary': 7,
                               'tertiary': 6, 'residential': 5, 'unclassified': 4,
                               'service': 3, 'road': 4}
                _n_roads = 0
                for _i, (_, row) in enumerate(gdf_roads.iterrows()):
                    geom = row.geometry
                    if geom is None or geom.is_empty:
                        continue
                    hw = str(row.get('highway', '')).lower()
                    width = _ROAD_WIDTH.get(hw, 4) / 2.0
                    lines = geom.geoms if isinstance(geom, sg.MultiLineString) else [geom]
                    for line in lines:
                        if not isinstance(line, sg.LineString):
                            continue
                        coords_local = [_to_local(lon, lat) for lon, lat in line.coords]
                        buffered = sg.LineString(coords_local).buffer(width, cap_style=2, join_style=2)
                        _n_roads += _append_clutter_polygon(buffered, ROAD_HEIGHT_M, 'osm_road', 'OSM_Road', _i)
                print(f'  Roads     : {_n_roads} segment mesh(es)')
                _osm_clutter_added += _n_roads
            except Exception as _e:
                print(f'  Roads download/build failed: {_e}')
        else:
            print('  Roads     : skipped (INCLUDE_OSM_ROADS=False)')

        # ── Vegetation ───────────────────────────────────────────────────────
        if INCLUDE_OSM_VEGETATION:
            try:
                gdf_veg = _features_from_bbox({'landuse': ['forest', 'orchard'],
                                                'natural': ['wood', 'scrub']})
                _n_veg = 0
                for _i, (_, row) in enumerate(gdf_veg.iterrows()):
                    geom = row.geometry
                    if geom is None or geom.is_empty:
                        continue
                    _n_veg += _append_clutter_polygon(geom, VEGETATION_HEIGHT_M, 'osm_vegetation', 'OSM_Vegetation', _i)
                print(f'  Vegetation: {_n_veg} polygon mesh(es)')
                _osm_clutter_added += _n_veg
            except Exception as _e:
                print(f'  Vegetation download/build failed: {_e}')
        else:
            print('  Vegetation: skipped (INCLUDE_OSM_VEGETATION=False)')

        # ── Water ──────────────────────────────────────────────────────────
        if INCLUDE_OSM_WATER:
            try:
                gdf_water = _features_from_bbox({'natural': 'water'})
                _n_water = 0
                for _i, (_, row) in enumerate(gdf_water.iterrows()):
                    geom = row.geometry
                    if geom is None or geom.is_empty:
                        continue
                    _n_water += _append_clutter_polygon(geom, WATER_HEIGHT_M, 'osm_water', 'OSM_Water', _i)
                print(f'  Water     : {_n_water} polygon mesh(es)')
                _osm_clutter_added += _n_water
            except Exception as _e:
                print(f'  Water download/build failed: {_e}')
        else:
            print('  Water     : skipped (INCLUDE_OSM_WATER=False)')

print(f'OSM clutter meshes added: {_osm_clutter_added}  (total usd_meshes: {len(usd_meshes)})')


## CELL 2d — Export usd_meshes -> Real .usd File

Writes every mesh currently in `usd_meshes` (USD/OSM buildings, DEM terrain, OSM roads/vegetation/water clutter) into an actual USD stage (`EXPORT_USD_FILE`), so the procedurally-built scene round-trips back into Omniverse/USD tooling as a real `.usd` asset -- not just the PLY/Mitsuba XML used by Sionna. Each mesh becomes a `UsdGeom.Mesh` prim under `/World`, with a `UsdShade.Material` (UsdPreviewSurface) bound to it carrying its ITU material's display colour, so the file previews sensibly in usdview/Omniverse without needing real textures. Requires `usd-core` (`pip install usd-core`); skips gracefully if unavailable or `EXPORT_USD=False`.

In [ ]:
# ============================================================
# CELL 2d — EXPORT usd_meshes -> scene_built.usda
# ============================================================
# Pure Python USDA ASCII writer — no pxr / usd-core needed.
# Groups meshes by ITU material (one Mesh prim per material).
# Open the output .usda in NVIDIA Omniverse, USD View, or Blender.
# ============================================================
import os
import numpy as np

_usd_exported = False

if not globals().get('EXPORT_USD', True):
    print('EXPORT_USD=False — skipping USD export.')
else:
    os.makedirs(os.path.dirname(EXPORT_USD_FILE), exist_ok=True)

    # ── Merge vertices/faces per ITU material ─────────────────────────────────
    _mat_verts = {}
    _mat_faces = {}
    _offsets   = {}
    for _m in usd_meshes:
        _mat = mesh_itu_material.get(_m['name'], _m.get('material', 'itu_concrete'))
        _vv  = np.asarray(_m['verts'], dtype=np.float32)
        _ff  = np.asarray(_m['faces'],  dtype=np.int32)
        if _mat not in _mat_verts:
            _mat_verts[_mat] = []; _mat_faces[_mat] = []; _offsets[_mat] = 0
        _mat_faces[_mat].append(_ff + _offsets[_mat])
        _offsets[_mat]  += len(_vv)
        _mat_verts[_mat].append(_vv)

    _merged = {
        _mat: {
            'verts': np.concatenate(_mat_verts[_mat], axis=0),
            'faces': np.concatenate(_mat_faces[_mat], axis=0),
        }
        for _mat in _mat_verts
    }

    # ── Display colours per ITU material ─────────────────────────────────────
    _RGB = {
        'itu_brick':             (0.65, 0.16, 0.16),
        'itu_concrete':          (0.55, 0.55, 0.55),
        'itu_glass':             (0.60, 0.85, 0.90),
        'itu_metal':             (0.70, 0.70, 0.75),
        'itu_wood':              (0.50, 0.35, 0.20),
        'itu_wet_ground':        (0.55, 0.42, 0.22),
        'itu_medium_dry_ground': (0.75, 0.65, 0.45),
        'itu_very_dry_ground':   (0.80, 0.72, 0.55),
        'itu_asphalt':           (0.18, 0.18, 0.18),
        'itu_vegetation':        (0.22, 0.55, 0.20),
        'itu_water':             (0.15, 0.35, 0.65),
    }
    def _rgb(mat):
        return _RGB.get(mat, (0.5, 0.5, 0.5))

    # ── Efficient USDA array formatters ───────────────────────────────────────
    def _int_list(arr):
        return '[' + ', '.join(map(str, arr.flatten().tolist())) + ']'

    def _pt3f_list(verts):
        v = np.round(verts.astype(np.float64), 3)
        rows = ['(' + ', '.join(map(str, r.tolist())) + ')' for r in v]
        return '[' + ', '.join(rows) + ']'

    # ── Write .usda ───────────────────────────────────────────────────────────
    print(f'Writing {EXPORT_USD_FILE} ...')
    with open(EXPORT_USD_FILE, 'w') as _f:

        # Header
        _f.write('#usda 1.0\n(\n')
        _f.write('    defaultPrim = "World"\n')
        _f.write('    upAxis = "Z"\n')
        _f.write('    metersPerUnit = 1.0\n')
        _f.write('    customLayerData = {\n')
        _f.write(f'        string anchorLon = "{ANCHOR_LON}"\n')
        _f.write(f'        string anchorLat = "{ANCHOR_LAT}"\n')
        _f.write(f'        string epsg      = "{UTM_EPSG}"\n')
        _f.write(f'        string scenario  = "{SCENARIO_NAME}"\n')
        _f.write('    }\n)\n\n')
        _f.write('def Xform "World"\n{\n\n')

        # Materials block
        _f.write('    # ── ITU-R P.2040 Materials ────────────────────────────────\n')
        for _mat in _merged:
            _s   = _mat.replace('-', '_')
            _r, _g, _b = _rgb(_mat)
            _f.write(f'    def Material "{_s}"\n    {{\n')
            _f.write(f'        def Shader "surface"\n        {{\n')
            _f.write(f'            uniform token info:id = "UsdPreviewSurface"\n')
            _f.write(f'            color3f inputs:diffuseColor = ({_r:.3f}, {_g:.3f}, {_b:.3f})\n')
            _f.write(f'            float inputs:roughness = 0.5\n')
            _f.write(f'            token outputs:surface\n')
            _f.write(f'        }}\n    }}\n\n')

        # Mesh block — one prim per ITU material
        _f.write('    # ── Meshes (one per ITU material) ─────────────────────────\n')
        for _mat, _data in _merged.items():
            _s     = _mat.replace('-', '_')
            _verts = _data['verts']
            _faces = _data['faces']
            print(f'  {_mat:<32} V={len(_verts):>7,}  F={len(_faces):>7,}')
            _f.write(f'    def Mesh "{_s}_mesh"\n    {{\n')
            _f.write(f'        int[] faceVertexCounts = [{", ".join(["3"]*len(_faces))}]\n')
            _f.write(f'        int[] faceVertexIndices = {_int_list(_faces)}\n')
            _f.write(f'        point3f[] points = {_pt3f_list(_verts)}\n')
            _f.write(f'        uniform bool doubleSided = 1\n')
            _f.write(f'        rel material:binding = </World/{_s}>\n')
            _f.write(f'    }}\n\n')

        _f.write('}\n')

    _fsize = os.path.getsize(EXPORT_USD_FILE) / 1024 / 1024
    _usd_exported = True
    _total_v = sum(len(d['verts']) for d in _merged.values())
    _total_f = sum(len(d['faces']) for d in _merged.values())
    print(f'\nExported : {EXPORT_USD_FILE}')
    print(f'  Size   : {_fsize:.1f} MB')
    print(f'  Prims  : {len(_merged)} mesh prims (one per ITU material)')
    print(f'  Total  : {_total_v:,} vertices  {_total_f:,} faces')
    print(f'  Open in: NVIDIA Omniverse / USD View / Blender (USD add-on)')


## CELL 3 — Map USD Materials → ITU-R P.2040-2 Material Names
USD/Omniverse materials carry visual (PBR/MaterialX) names, not RF electromagnetic
properties — there is no built-in "this is concrete for radio purposes" semantic.
This cell matches each USD material's **name** against the same keyword set the
OSM pipeline uses, falling back to a default if nothing matches. If your USD
materials use opaque names (e.g. `Material_007`), you'll need to either rename
them before export or extend `_USD_MAT_KEYWORDS` with project-specific aliases —
matching by `diffuseColor`/MaterialX graph instead of name is a possible future
upgrade, not implemented here.

In [ ]:
# ============================================================
# CELL 3 — USD MATERIAL NAME → ITU-R MATERIAL MAPPING
# ============================================================
_USD_MAT_KEYWORDS = {
    'itu_concrete'  : ('concrete', 'cement', 'paving'),
    'itu_brick'     : ('brick',),
    'itu_glass'     : ('glass', 'window', 'curtain_wall', 'curtainwall'),
    'itu_metal'     : ('metal', 'steel', 'aluminium', 'aluminum', 'cladding'),
    'itu_wood'      : ('wood', 'timber', 'plywood'),
    'itu_plasterboard': ('plaster', 'drywall', 'gypsum'),
    'itu_marble'    : ('marble',),
    'itu_asphalt'   : ('asphalt', 'tarmac', 'road'),
    'itu_vegetation': ('vegetation', 'tree', 'foliage', 'grass', 'leaf'),
    'itu_water'     : ('water', 'pond', 'river', 'lake'),
    'itu_wet_ground': ('ground', 'soil', 'dirt', 'terrain'),
}
_DEFAULT_ITU_MAT = 'itu_concrete'

def _match_usd_material(usd_mat_name):
    n = usd_mat_name.lower()
    for itu_name, kws in _USD_MAT_KEYWORDS.items():
        if any(kw in n for kw in kws):
            return itu_name
    return None

print('USD material -> ITU mapping:')
mesh_itu_material = {}
for m in usd_meshes:
    matched = _match_usd_material(m['material'])
    itu_name = matched or _DEFAULT_ITU_MAT
    mesh_itu_material[m['name']] = itu_name
    flag = '' if matched else '  (no keyword match -- using default)'
    print(f"  {m['material']:<24} -> {itu_name:<16}{flag}")


## CELL 3b — NVIDIA NIM Vision-Language Material Classification (optional)

Overrides the filename-keyword ITU material guess from CELL 3 with an actual visual classification of each building's real-world facade, by sending a satellite-imagery crop of its footprint to an NVIDIA NIM-hosted vision-language model (`build.nvidia.com`). Off by default (`USE_NVIDIA_MATERIAL_CLASSIFICATION`); needs `NVIDIA_API_KEY` set and network access to both the NIM endpoint and the satellite tile server. Falls back silently to the CELL 3 keyword guess per-mesh on any failure (missing key, network error, unparseable response).

In [ ]:
# ============================================================
# CELL 3b — NVIDIA NIM VISION-LANGUAGE MATERIAL CLASSIFICATION (optional)
# ============================================================
_nvidia_classified = 0

if not USE_NVIDIA_MATERIAL_CLASSIFICATION:
    print('Skipping NVIDIA NIM material classification -- USE_NVIDIA_MATERIAL_CLASSIFICATION=False.')
elif not USD_HAS_GEOREFERENCE:
    print('Skipping NVIDIA NIM material classification -- USD_HAS_GEOREFERENCE=False (no real-world bbox).')
elif not NVIDIA_API_KEY:
    print('Skipping NVIDIA NIM material classification -- NVIDIA_API_KEY not set. '
          'Get one at https://build.nvidia.com and set the NVIDIA_API_KEY env var.')
else:
    import requests, base64
    from pyproj import Transformer

    _wgs_to_bng3 = Transformer.from_crs('EPSG:4326', f'EPSG:{UTM_EPSG}', always_xy=True)
    _bng_to_wgs3 = Transformer.from_crs(f'EPSG:{UTM_EPSG}', 'EPSG:4326', always_xy=True)
    _anchor_e3, _anchor_n3 = _wgs_to_bng3.transform(ANCHOR_LON, ANCHOR_LAT)

    _VLM_MATERIAL_OPTIONS = ('itu_brick', 'itu_concrete', 'itu_glass', 'itu_metal',
                              'itu_wood', 'itu_marble')

    def _fetch_satellite_crop(cx, cy, half_extent_m=30.0, px=256):
        """Esri World Imagery export crop around a local (x, y) point, returned as PNG bytes.
        Shows the roof, not the facade -- used as a fallback when street-level
        imagery isn't available."""
        e0, n0 = _anchor_e3 + cx - half_extent_m, _anchor_n3 + cy - half_extent_m
        e1, n1 = _anchor_e3 + cx + half_extent_m, _anchor_n3 + cy + half_extent_m
        w_lon, s_lat = _bng_to_wgs3.transform(e0, n0)
        e_lon, n_lat = _bng_to_wgs3.transform(e1, n1)
        params = {'bbox': f'{w_lon},{s_lat},{e_lon},{n_lat}', 'bboxSR': 4326,
                  'size': f'{px},{px}', 'imageSR': 3857, 'format': 'png', 'f': 'image'}
        r = requests.get(SATELLITE_TILE_URL, params=params, timeout=20)
        r.raise_for_status()
        return r.content

    def _fetch_streetview_crop(cx, cy, radius_m=50.0):
        """Nearest Mapillary street-level image to a local (x, y) point, returned as
        JPEG bytes -- shows the actual facade, not the roof. Returns None if no
        image is found nearby or the token isn't set."""
        if not MAPILLARY_TOKEN:
            return None
        e, n = _anchor_e3 + cx, _anchor_n3 + cy
        lon, lat = _bng_to_wgs3.transform(e, n)
        r = requests.get('https://graph.mapillary.com/images', params={
            'access_token': MAPILLARY_TOKEN, 'fields': 'id,thumb_1024_url',
            'closeto': f'{lon},{lat}', 'radius': radius_m, 'limit': 1}, timeout=20)
        r.raise_for_status()
        data = r.json().get('data', [])
        if not data:
            return None
        img_url = data[0].get('thumb_1024_url')
        if not img_url:
            return None
        ir = requests.get(img_url, timeout=20)
        ir.raise_for_status()
        return ir.content

    def _fetch_material_crop(cx, cy):
        """Street-level facade crop (preferred) with a satellite roof crop fallback,
        per MATERIAL_IMAGE_SOURCE."""
        if MATERIAL_IMAGE_SOURCE == 'streetview':
            crop = _fetch_streetview_crop(cx, cy)
            if crop is not None:
                return crop, 'streetview'
        return _fetch_satellite_crop(cx, cy), 'satellite'

    def _classify_material_nvidia_nim(png_bytes):
        """Ask an NVIDIA NIM vision-language model which ITU material best matches
        the building facade in the image. Returns an itu_* name, or None on failure."""
        b64 = base64.b64encode(png_bytes).decode('ascii')
        prompt = (
            'Look at this satellite image of a single building. Reply with exactly one '
            'word from this list, the material that best matches the building roof/facade: '
            + ', '.join(m.replace('itu_', '') for m in _VLM_MATERIAL_OPTIONS))
        payload = {
            'model': NVIDIA_NIM_VLM_MODEL,
            'messages': [{'role': 'user', 'content': [
                {'type': 'text', 'text': prompt},
                {'type': 'image_url', 'image_url': {'url': f'data:image/png;base64,{b64}'}},
            ]}],
            'max_tokens': 16,
            'temperature': 0.0,
        }
        headers = {'Authorization': f'Bearer {NVIDIA_API_KEY}', 'Accept': 'application/json'}
        r = requests.post(f'{NVIDIA_NIM_BASE_URL}/chat/completions',
                           json=payload, headers=headers, timeout=30)
        r.raise_for_status()
        text = r.json()['choices'][0]['message']['content'].strip().lower()
        for m in _VLM_MATERIAL_OPTIONS:
            if m.replace('itu_', '') in text:
                return m
        return None

    for m in usd_meshes:
        if m['name'] in ('dem_terrain', 'ground') or m['name'].startswith(('osm_road_', 'osm_water_')):
            continue  # only classify real buildings, not terrain/road/water meshes
        try:
            cx, cy = m['verts'][:, 0].mean(), m['verts'][:, 1].mean()
            crop, source = _fetch_material_crop(cx, cy)
            classified = _classify_material_nvidia_nim(crop)
            if classified:
                mesh_itu_material[m['name']] = classified
                _nvidia_classified += 1
                print(f"  {m['name']:<24} -> {classified}  (NVIDIA NIM, {source} imagery)")
        except Exception as _e:
            print(f"  {m['name']:<24} -> classification failed ({_e}) -- keeping CELL 3 keyword guess")

print(f'NVIDIA NIM material classification: {_nvidia_classified} mesh(es) reclassified')


## CELL 4 — Write Per-Mesh PLYs
Same writer (`_write_ply`) the OSM builder's CELL 4 uses, so downstream tooling
(CELL 5's `scene.xml` writer, `blender_to_sionna2_converter.py`, the simulation
notebooks) sees identical PLY files regardless of where the geometry came from.

In [ ]:
# ============================================================
# CELL 4 — WRITE PLYs
# ============================================================
def _write_ply(verts, faces, path):
    verts = np.asarray(verts, dtype=np.float32)
    faces = np.asarray(faces, dtype=np.int32)
    if _HAS_TRIMESH:
        trimesh.Trimesh(vertices=verts, faces=faces, process=False).export(path)
        return
    with open(path, 'w') as f:
        f.write('ply\nformat ascii 1.0\n')
        f.write(f'element vertex {len(verts)}\n')
        f.write('property float x\nproperty float y\nproperty float z\n')
        f.write(f'element face {len(faces)}\n')
        f.write('property list uchar int vertex_indices\nend_header\n')
        for v in verts:  f.write(f'{v[0]:.4f} {v[1]:.4f} {v[2]:.4f}\n')
        for fc in faces: f.write(f'3 {fc[0]} {fc[1]} {fc[2]}\n')

# mat_plys: {itu_material_name: [(relative_ply_path, role), ...]} -- same shape
# the OSM builder's CELL 5 scene.xml writer expects.
mat_plys = {}
ground_ply = None

for m in usd_meshes:
    itu_name = mesh_itu_material[m['name']]
    fname = f"usd_{m['name']}.ply"
    fpath = os.path.join(MESH_DIR, fname)
    _write_ply(m['verts'], m['faces'], fpath)
    rel_path = os.path.basename(MESH_DIR) + '/' + fname

    # 'dem_terrain' = real EA LiDAR terrain built in CELL 2b; 'ground' = the
    # flat synthetic ground slab (demo mode, or no DEM terrain available).
    is_ground = m['name'] in ('ground', 'dem_terrain')
    if is_ground:
        ground_ply = rel_path
        continue
    mat_plys.setdefault(itu_name, []).append((rel_path, 'usd_import'))

print(f'Wrote {len(usd_meshes)} PLY(s) -> {MESH_DIR}')
print(f'Ground PLY : {ground_ply or "(none found -- CELL 5 will fall back to a flat terrain.ply)"}')
print(f'mat_plys   : {{ {", ".join(f"{k}: {len(v)}" for k, v in mat_plys.items())} }}')

# CELL 5 (ported from the OSM builder) writes a fixed "terrain.ply" shape --
# reuse whichever ground/terrain mesh CELL 2/2b produced so the scene doesn't
# end up with two overlapping ground planes.
import shutil
terrain_ply_path = os.path.join(MESH_DIR, 'terrain.ply')
if ground_ply is not None:
    shutil.copyfile(os.path.join(SCENE_DIR, ground_ply), terrain_ply_path)
    print(f'Copied USD/DEM ground mesh -> {terrain_ply_path}')
elif not os.path.exists(terrain_ply_path):
    _gv, _gf = _box_mesh(0, 0, 0, 1000, 1000, 0.0)
    _write_ply(_gv, _gf, terrain_ply_path)
    print(f'No ground mesh in USD scene -- wrote flat 1km x 1km placeholder terrain.ply')


## CELL 5 — Write scene.xml (Sionna 0.19) + scene_with_full.xml (Sionna 2.0)

Identical to `sionna019_scene_builder_london.ipynb` CELL B1 + CELL B3 combined:

| Output file | Format | Consumed by |
|-------------|--------|-------------|
| `scene.xml` | Sionna 0.19 Mitsuba (`diffuse` BSDFs) | `sionna019_differentiable_rt_fixed.ipynb` |
| `scene_with_full.xml` | Sionna 2.0 (`radio-material` BSDFs, `mat-` prefix) | `sionna2_915mhz_dem_simulation_london.ipynb` |

Only the source of `mat_plys` differs (USD/OSM meshes vs OSM footprints in the OSM builder),
so the XML output format is identical.

In [ ]:
# ============================================================
# CELL 5 — WRITE SCENE XML (Sionna 0.19 + Sionna 2.0)
# ============================================================
# Same as sionna019_scene_builder_london.ipynb CELL B1 + CELL B3:
#   scene.xml            -> Sionna 0.19 Mitsuba format  (diffuse BSDFs)
#   scene_with_full.xml  -> Sionna 2.0 format  (radio-material BSDFs, mat- prefix)
# Only the source of mat_plys differs (USD/OSM meshes here, OSM footprints there).
# ============================================================
import os, json as _json

TERRAIN_MATERIAL = globals().get('TERRAIN_MATERIAL', 'itu_wet_ground')

_center_lat_s2 = globals().get('ANCHOR_LAT', 51.5305)
_center_lon_s2 = globals().get('ANCHOR_LON', -0.13399)

# ── ITU-R P.2040-2 material table ────────────────────────────────────────────
_ITU_P2040_PARAMS = {
    #                       a       b       c        d      s     xpd
    'itu_concrete'    : (5.31,  0.000,  0.0326, 0.8095, 0.30, 0.10),
    'itu_brick'       : (3.91,  0.000,  0.0238, 0.0000, 0.25, 0.10),
    'itu_glass'       : (6.27,  0.000,  0.0043, 1.1925, 0.10, 0.05),
    'itu_plywood'     : (1.99,  0.000,  0.0047, 1.0718, 0.20, 0.10),
    'itu_metal'       : (1.00,  0.000,  1.0e7,  0.0000, 0.05, 0.05),
    'itu_wet_ground'          : (30.0, -0.400, 0.1500, 1.3000, 0.35, 0.05),
    'itu_water'               : (80.0,  0.000, 0.0100, 0.0000, 0.02, 0.05),
    'itu_medium_dry_ground'   : (15.0, -0.100, 0.0350, 1.6300, 0.10, 0.05),
    'itu_very_dry_ground'     : ( 3.0,  0.000, 0.00015,2.5200, 0.10, 0.05),
    'itu_vegetation'          : ( 1.50, 0.000, 0.0020, 0.5000, 0.40, 0.50),
    'itu_asphalt'             : ( 2.56, 0.000, 0.0050, 0.0000, 0.30, 0.15),
}
_f_ghz = float(globals().get('FREQUENCY_HZ', 915.95e6)) / 1e9
ITU_MATERIALS = {_mn: (round(_a * (_f_ghz**_b), 6), round(_c * (_f_ghz**_d), 8), _s, _xpd)
                 for _mn, (_a, _b, _c, _d, _s, _xpd) in _ITU_P2040_PARAMS.items()}

# ── Material remaps ───────────────────────────────────────────────────────────
# 0.19 remap: non-standard -> nearest Sionna 0.19 ITU type
_MAT_REMAP_019 = {
    'itu_wood'        : 'itu_plywood',
    'itu_water'       : 'itu_medium_dry_ground',
    'itu_vegetation'  : 'itu_ceiling_board',
    'itu_asphalt'     : 'itu_very_dry_ground',
    'itu_marble'      : 'itu_concrete',
    'itu_plasterboard': 'itu_ceiling_board',
}
# 2.0 remap: same intent, ceiling_board is valid in Sionna 2.0 registry
_NON_ITU_MAP_S2 = {
    'itu_asphalt'      : 'itu_very_dry_ground',
    'itu_vegetation'   : 'itu_ceiling_board',
    'itu_water'        : 'itu_medium_dry_ground',
    'itu_wood'         : 'itu_plywood',
    'itu_marble'       : 'itu_concrete',
    'itu_plasterboard' : 'itu_ceiling_board',
}
def _remap_019(m):  return _MAT_REMAP_019.get(m, m)
def _remap_s2(m):   return _NON_ITU_MAP_S2.get(m, m)

# ── Visual colours (Mitsuba rendering) ───────────────────────────────────────
_ITU_COLOURS = {
    'itu_concrete'          : '0.539 0.539 0.539',
    'itu_brick'             : '1.000 0.498 0.055',
    'itu_glass'             : '0.596 0.875 0.541',
    'itu_plywood'           : '0.514 0.376 0.220',
    'itu_metal'             : '0.220 0.220 0.254',
    'itu_wet_ground'        : '0.910 0.569 0.055',
    'itu_very_dry_ground'   : '0.498 0.498 0.498',
    'itu_medium_dry_ground' : '0.780 0.780 0.780',
    'itu_ceiling_board'     : '0.180 0.450 0.180',
}

used_mats = set(mat_plys.keys()) | {TERRAIN_MATERIAL}

# ============================================================
# A) SIONNA 0.19  (scene.xml)
# ============================================================
_used_019 = {_remap_019(m) for m in used_mats} | {TERRAIN_MATERIAL}
L019 = ['<?xml version="1.0" encoding="utf-8"?>', '<scene version="2.1.0">', '',
        '  <!-- ── ITU-R P.2040-2 Materials ────────────────────── -->']
for _mn in sorted(_used_019):
    _rgb = _ITU_COLOURS.get(_mn, '0.5 0.5 0.5')
    L019 += [f'  <bsdf type="diffuse" id="{_mn}">',
             f'    <rgb name="reflectance" value="{_rgb}"/>',
             '  </bsdf>', '']
L019 += ['  <!-- ── Terrain ─────────────────────────────────────── -->',
         '  <shape type="ply" id="mesh-ground">',
         '    <string name="filename" value="meshes/terrain.ply"/>',
         f'    <ref id="{TERRAIN_MATERIAL}" name="bsdf"/>',
         '    <boolean name="face_normals" value="true"/>',
         '  </shape>', '',
         '  <!-- ── Scene geometry ───────────────────────────────── -->']
for _mn, _pl in sorted(mat_plys.items()):
    _ref = _remap_019(_mn)
    for _pp, _ in _pl:
        _mid = 'mesh-' + _pp.split('/')[-1].replace('.ply', '')
        L019 += [f'  <shape type="ply" id="{_mid}">',
                 f'    <string name="filename" value="{_pp}"/>',
                 f'    <ref id="{_ref}" name="bsdf"/>',
                 '    <boolean name="face_normals" value="true"/>',
                 '  </shape>']
L019 += ['', '</scene>']
_xml_019 = os.path.join(SCENE_DIR, 'scene.xml')
with open(_xml_019, 'w') as _f:
    _f.write('\n'.join(L019))
print(f'Sionna 0.19 XML : {_xml_019}')

# ============================================================
# B) SIONNA 2.0  (scene_with_full.xml)
# ============================================================
_final_ids_s2 = sorted({_remap_s2(m) for m in used_mats})
LS2 = []
LS2.append('<?xml version="1.0" ?>')
LS2.append('<scene version="2.1.0">')
LS2.append('')

LS2.append('  <!-- ── Scene metadata ──────────────────────────────────── -->')
LS2.append('  <default name="scenegen_version"     value="1.0.0"/>')
LS2.append(f'  <default name="scenegen_min_lat"     value="{globals().get(\'SCENE_SOUTH\', \"\")}"/>')
LS2.append(f'  <default name="scenegen_max_lat"     value="{globals().get(\'SCENE_NORTH\', \"\")}"/>')
LS2.append(f'  <default name="scenegen_min_lon"     value="{globals().get(\'SCENE_WEST\', \"\")}"/>')
LS2.append(f'  <default name="scenegen_max_lon"     value="{globals().get(\'SCENE_EAST\', \"\")}"/>')
LS2.append(f'  <default name="scenegen_center_lat"  value="{_center_lat_s2}"/>')
LS2.append(f'  <default name="scenegen_center_lon"  value="{_center_lon_s2}"/>')
LS2.append(f'  <default name="scenegen_UTM_zone"    value="EPSG:{UTM_EPSG}"/>')
LS2.append(f'  <default name="scenegen_ground_material"  value="mat-{TERRAIN_MATERIAL}"/>')
LS2.append('')

LS2.append('  <!-- ── Integrator ──────────────────────────────────────── -->')
LS2.append('  <integrator type="path">')
LS2.append('    <integer name="max_depth" value="12"/>')
LS2.append('  </integrator>')
LS2.append('')

LS2.append('  <!-- ── ITU-R Materials (radio-material for Sionna 2.0) ───── -->')
for _fid in _final_ids_s2:
    LS2.append(f'  <bsdf type="radio-material" id="mat-{_fid}"/>')
    LS2.append('')

LS2.append('  <!-- ── Environment ─────────────────────────────────────── -->')
LS2.append('  <emitter type="constant" id="World">')
LS2.append('    <rgb value="1.0 1.0 1.0" name="radiance"/>')
LS2.append('  </emitter>')
LS2.append('')

LS2.append('  <!-- ── Camera (top-down overview) ─────────────────────── -->')
LS2.append('  <sensor type="perspective" id="Camera">')
LS2.append('    <string name="fov_axis" value="x"/>')
LS2.append('    <float  name="fov"      value="42.855"/>')
LS2.append('    <float  name="near_clip" value="0.1"/>')
LS2.append('    <float  name="far_clip"  value="10000.0"/>')
LS2.append('    <transform name="to_world">')
LS2.append('      <rotate z="1" angle="-90"/>')
LS2.append('      <translate value="0 0 500"/>')
LS2.append('    </transform>')
LS2.append('    <sampler type="independent">')
LS2.append('      <integer name="sample_count" value="4096"/>')
LS2.append('    </sampler>')
LS2.append('    <film type="hdrfilm">')
LS2.append('      <integer name="width"  value="1024"/>')
LS2.append('      <integer name="height" value="1024"/>')
LS2.append('    </film>')
LS2.append('  </sensor>')
LS2.append('')

LS2.append('  <!-- ── Terrain ─────────────────────────────────────────── -->')
LS2.append('  <shape type="ply" id="mesh-ground">')
LS2.append('    <string name="filename" value="meshes/terrain.ply"/>')
LS2.append(f'    <ref id="mat-{TERRAIN_MATERIAL}" name="bsdf"/>')
LS2.append('    <boolean name="face_normals" value="true"/>')
LS2.append('  </shape>')
LS2.append('')

LS2.append('  <!-- ── Scene geometry (buildings, roads, clutter) ───────── -->')
for _mn, _pl in sorted(mat_plys.items()):
    for _pp, _role in _pl:
        _mid = 'mesh-' + _pp.split('/')[-1].replace('.ply', '')
        LS2.append(f'  <shape type="ply" id="{_mid}">')
        LS2.append(f'    <string name="filename" value="{_pp}"/>')
        LS2.append(f'    <ref id="mat-{_remap_s2(_mn)}" name="bsdf"/>')
        LS2.append('    <boolean name="face_normals" value="true"/>')
        LS2.append('  </shape>')

LS2.append('')
LS2.append('</scene>')

SIONNA2_XML_OUT = os.path.join(SCENE_DIR, 'scene_with_full.xml')
with open(SIONNA2_XML_OUT, 'w') as _f:
    _f.write('\n'.join(LS2))

_n_shapes = sum(len(v) for v in mat_plys.values())
print(f'Sionna 2.0 XML  : {SIONNA2_XML_OUT}')
print(f'  Materials     : {len(used_mats)}  -> {len(_final_ids_s2)} Sionna 2.0 types')
print(f'  Shapes        : 1 terrain + {_n_shapes} geometry mesh(es)')

# ── scene_parameters.json ──────────────────────────────────────────────────────
_meta = {
    'utm_epsg': UTM_EPSG, 'projection_crs': PROJECTION_CRS,
    'source': 'omniverse_osm',
    'demo_mode': globals().get('DEMO_MODE', False),
    'n_meshes': len(usd_meshes),
    'scene_center_lat': _center_lat_s2,
    'scene_center_lon': _center_lon_s2,
    'scene_west'  : globals().get('SCENE_WEST',  ''),
    'scene_east'  : globals().get('SCENE_EAST',  ''),
    'scene_south' : globals().get('SCENE_SOUTH', ''),
    'scene_north' : globals().get('SCENE_NORTH', ''),
    'frequency_hz': globals().get('FREQUENCY_HZ', 915.95e6),
    'sionna2_xml' : SIONNA2_XML_OUT,
    'sionna019_xml': _xml_019,
}
with open(os.path.join(BASE_DIR, 'scene_parameters.json'), 'w') as _f:
    _json.dump(_meta, _f, indent=2)
print(f'Metadata        : {os.path.join(BASE_DIR, "scene_parameters.json")}')


## CELL 6 — 2D Top-Down Preview

In [ ]:
# ============================================================
# CELL 6 — 2D PREVIEW OF EXTRACTED GEOMETRY
# ============================================================
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(8, 8))
_cmap = plt.get_cmap('tab10')
_mat_colors = {m: _cmap(i % 10) for i, m in enumerate(sorted({mesh_itu_material[m['name']] for m in usd_meshes}))}

for m in usd_meshes:
    v = m['verts']
    xs, ys = v[:, 0], v[:, 1]
    if m['name'] == 'dem_terrain':
        # Dense N x N grid mesh -- angle-sort-and-fill produces a starburst
        # of crossing lines on a grid (it only works for small convex
        # footprints like the building boxes below). Draw it as a flat
        # extent rectangle + a light hatch instead, so it doesn't swamp
        # the actual buildings.
        ax.add_patch(mpatches.Rectangle(
            (xs.min(), ys.min()), xs.max() - xs.min(), ys.max() - ys.min(),
            facecolor=_mat_colors[mesh_itu_material[m['name']]], alpha=0.15,
            edgecolor='none', zorder=0))
        continue
    hull_order = np.argsort(np.arctan2(ys - ys.mean(), xs - xs.mean()))
    ax.fill(xs[hull_order], ys[hull_order], alpha=0.5,
            color=_mat_colors[mesh_itu_material[m['name']]], edgecolor='k', linewidth=0.5,
            zorder=2)
    ax.text(xs.mean(), ys.mean(), m['name'], fontsize=7, ha='center', zorder=3)

handles = [mpatches.Patch(color=c, label=m) for m, c in _mat_colors.items()]
ax.legend(handles=handles, loc='upper right', fontsize=8)
ax.set_xlabel('Local X (m)'); ax.set_ylabel('Local Y (m)')
ax.set_title(f'USD scene preview ({"DEMO" if DEMO_MODE else USD_FILE})  --  {len(usd_meshes)} prims')
ax.set_aspect('equal', adjustable='datalim')
plt.tight_layout()
_png = os.path.join(SCENE_DIR, 'usd_scene_preview.png')
plt.savefig(_png, dpi=130)
plt.close()
try:
    from IPython.display import Image, display
    display(Image(filename=_png, width=700))
except Exception:
    pass
print(f'Saved preview -> {_png}')


## CELL 7 — Load into Sionna RT and Sanity-Check
Requires Sionna to be installed; safe to skip if you only want to inspect the
written `scene.xml`/PLYs.

In [ ]:
# ============================================================
# CELL 7 — LOAD INTO SIONNA RT
# ============================================================
try:
    import sionna
    from sionna.rt import load_scene
    print(f'Sionna : {sionna.__version__}')
    scene = load_scene(scene_xml)
    print(f'Loaded : {scene_xml}')
    print(f'Radio materials ({len(scene.radio_materials)}): {sorted(scene.radio_materials.keys())}')
    print(f'Scene objects ({len(scene.objects)}): {sorted(scene.objects.keys())}')
    bbox = scene.mi_scene.bbox()
    print(f'BBox   : min={list(bbox.min)}  max={list(bbox.max)}')
except ImportError as _e:
    print(f'Sionna not installed in this environment -- skipping load check ({_e})')
except Exception as _e:
    print(f'Scene load FAILED: {_e}')
    import traceback; traceback.print_exc()
